# 03 — Train

Trains the Siamese U-Net with AMP, cosine schedule and early stopping on val mIoU. After every epoch the checkpoints are pushed to Drive — **if Colab disconnects, just re-run this notebook and training resumes** from `last.ckpt` (at most one epoch is lost).

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # the whole 00-07 series runs this branch
if os.path.exists(REPO):
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.train import train

train("configs/train.yaml", repo_root=".", store=store)


In [ ]:
import json
import matplotlib.pyplot as plt

run_dir = store.fetch_run("exp1")
history = json.loads((run_dir / "history.json").read_text())

best_ep = max(range(len(history)), key=lambda i: history[i]["val_miou"])
print(f"best epoch {best_ep + 1}/{len(history)} - val mIoU "
      f"{history[best_ep]['val_miou']:.4f} (early stop by patience)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot([h["train_loss"] for h in history], label="train")
ax1.plot([h["val_loss"] for h in history], label="val")
ax1.set_title("Training loss (cross-entropy + Dice)")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend()
ax2.plot([h["val_miou"] for h in history], label="mIoU (mean of both classes)")
ax2.plot([h["val_change_iou"] for h in history], label="change IoU (the hard class)")
ax2.axvline(best_ep, color="gray", ls="--", lw=1, label=f"best epoch ({best_ep + 1})")
ax2.set_title("Validation performance by epoch")
ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout()
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
fig.savefig(rep / "training_curves.png", dpi=150, bbox_inches="tight")
store.push_report("training_curves.png")
print("pushed training_curves.png -> Drive")
plt.show()
